# Parte 3: visualização do episódio de seca

Os gráficos da seção 10 do notebook `02_dqn` mostram o episódio de seca inteiro de uma vez. Aqui o mesmo episódio vira uma **página animada**, que mostra a estufa hora a hora: o clima, a umidade de cada canteiro, o tanque, a decisão da política e a recompensa.

São dois passos, cada um num script da pasta `scripts/`:

1. `gerar_episodios.py` roda o episódio de seca com o termostato e com o DQN treinado e grava tudo num arquivo JSON;
2. `montar_pagina.py` coloca esse JSON dentro do modelo da página (`web/estufa_modelo.html`) e grava `docs/estufa/index.html`.

Nada é treinado aqui: o DQN é o modelo salvo em `resultados/1000k_s4242/dqn_final.zip`, o mesmo do notebook `02_dqn`.


## Preparação

O código do simulador fica no arquivo `estufa.py`, na raiz do repositório, para ser usado pelos três notebooks. A célula abaixo clona o repositório (se ainda não estiver clonado), põe a raiz dele no `sys.path` e instala as bibliotecas.


In [ ]:
import os, sys

if os.path.exists("estufa.py"):                  # já está na raiz do repositório
    RAIZ = os.getcwd()
elif os.path.exists("../estufa.py"):             # rodando de dentro de notebooks/
    RAIZ = os.path.abspath("..")
else:                                            # Colab: clona o repositório
    if not os.path.exists("agent-dedoverde"):
        !git clone -q https://github.com/gabrielhca/agent-dedoverde.git
    RAIZ = os.path.abspath("agent-dedoverde")
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
print("raiz do repositório:", RAIZ)

!pip install -q gymnasium stable-baselines3


## 1. Gerar o episódio

O script usa a **semente 1**, a mesma do `SEED_SECA` do notebook `02_dqn`, com o `EstufaEnv()` padrão. Como a semente é a mesma, o clima é idêntico para as duas políticas.

Para cada hora ele guarda uma linha com 17 valores:

| Colunas | O que são |
|---|---|
| `t`, `T_in`, `T_ext`, `M_milho`, `M_caupi`, `tanque`, `viva_milho`, `viva_caupi`, `sol`, `chuva`, `prev` | o estado da estufa **depois** do passo (vem do `info` do ambiente) |
| `irr_milho`, `irr_caupi`, `aquec`, `vent` | a ação escolhida, já decodificada (`env.decodificar`) |
| `r`, `G` | a recompensa da hora e a soma acumulada até ali |

Só para o DQN, o script guarda também os **36 valores Q** que a rede calculou antes de cada decisão, e confere que a ação escolhida é a de maior Q (é isso que `deterministic=True` faz).

As linhas impressas abaixo têm de bater com a saída da seção 10 do notebook `02_dqn`.


In [2]:
!python "{RAIZ}/scripts/gerar_episodios.py"


Termostato: retorno 111.1 | água 1750 L | 470 horas | mortes: milho no dia 18.5, feijão-caupi no dia 19.6


DQN: retorno 83.8 | água 1800 L | 449 horas | mortes: milho no dia 17.4, feijão-caupi no dia 18.7


gravado: resultados/episodios/episodio_1.json


Uma olhada no arquivo gerado: as primeiras horas do DQN e os valores Q da primeira decisão.


In [3]:
import json
import numpy as np
import pandas as pd

with open(os.path.join(RAIZ, "resultados", "episodios", "episodio_1.json"), encoding="utf-8") as f:
    dados = json.load(f)

dqn = dados["eps"][1]
display(pd.DataFrame(dqn["h"], columns=dados["cols"]).head())

q0 = np.array(dqn["q"][0])
print(f"{len(dqn['h'])} horas | {q0.size} valores Q por hora")
print(f"1ª decisão: maior Q = {q0.max():.2f} (ação {q0.argmax()}), menor Q = {q0.min():.2f}")


,t,T_in,T_ext,M_milho,M_caupi,tanque,viva_milho,viva_caupi,sol,chuva,prev,irr_milho,irr_caupi,aquec,vent,r,G
0,1,25.55,11.40,79.20,62.92,1349.5,1,1,0.0,0.0,0.117,0,0,1.0,0.0,0.7000,0.700
1,2,23.43,10.80,78.67,62.62,1349.5,1,1,0.0,0.0,0.000,0,0,0.0,0.0,0.7877,1.488
2,3,24.53,10.56,78.09,62.29,1349.5,1,1,0.0,0.0,0.000,0,0,1.0,0.0,0.7000,2.188
3,4,25.43,10.84,77.46,61.93,1349.5,1,1,0.0,0.0,0.000,0,0,1.0,0.0,0.7000,2.888
4,5,26.25,11.57,76.79,61.55,1349.5,1,1,0.0,0.0,0.178,0,0,1.0,0.0,0.7000,3.588


449 horas | 36 valores Q por hora
1ª decisão: maior Q = 78.81 (ação 6), menor Q = 75.24


## 2. Montar a página

O modelo `web/estufa_modelo.html` tem o marcador `__DADOS__` no lugar dos dados. O script troca o marcador pelo conteúdo do JSON e grava `docs/estufa/index.html`: um arquivo único, que abre direto no navegador.


In [4]:
!python "{RAIZ}/scripts/montar_pagina.py"


gravado: docs/estufa/index.html (199 KB)


## 3. Ver a página

- **Publicada:** https://gabrielhca.github.io/agent-dedoverde/estufa/
- **Na sua máquina:** abra o arquivo `docs/estufa/index.html` no navegador.
- **No Colab:** a célula abaixo mostra a página aqui mesmo.

Na página, os botões **Termostato** e **DQN** trocam a política. No DQN aparece o painel "O que a rede pensa", com os valores Q de cada hora.


In [ ]:
from IPython.display import HTML

with open(os.path.join(RAIZ, "docs", "estufa", "index.html"), encoding="utf-8") as f:
    pagina = f.read()
HTML(pagina)
